# ZoKrates / Groth16 measurement session on Colab (native mode)
Run the cells **from the top, one at a time** (Shift+Enter). Runtime > Change runtime type must be **CPU**. Leave this browser tab open while the session runs (cell 5).
Expected time: about 3 to 4 hours. Details: LINUX.md in the package.

In [ ]:
# 1. Upload the package (a button appears): choose zkp_v4d.zip
import os, zipfile
from google.colab import files
up = files.upload()
z = next(f for f in up if f.endswith('.zip'))
zipfile.ZipFile(z).extractall('/content')
os.chdir('/content/zkp_v4')
print(sorted(os.listdir('.')))

In [ ]:
# 2. Connect Google Drive: results are copied there after every pass; an interrupted session is continued from that copy
from google.colab import drive
drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/zkp_v4_colab'
os.makedirs(BACKUP, exist_ok=True)
!cp -rn "$BACKUP"/. /content/zkp_v4/ 2>/dev/null; ls -d results_* memory_* native_* 2>/dev/null

In [ ]:
# 3. Two-minute functional check (it must end with 'READY' and then 'All steps finished', and no line may start with !!!)
!bash check_setup_linux.sh && bash run_all_linux.sh quick 2>&1 | tail -30

In [ ]:
# 4. Start the full session in the background (it continues an unfinished session by itself)
os.environ['AFTER_PASS_HOOK'] = f'cp -r "$RES" "{BACKUP}/"; cp -f run_all.out "{BACKUP}/" 2>/dev/null; true'
!bash start_linux.sh

In [ ]:
# 5. Watch it (prints every 5 minutes; keep this tab open until it says finished)
import subprocess, time
from IPython.display import clear_output
def running(): return subprocess.run(['pgrep', '-f', 'run_all_linux.sh'], capture_output=True).returncode == 0  # argument list, so the shell wrapper does not match itself
while running():
    out = subprocess.run('bash status_linux.sh | head -6', shell=True, capture_output=True, text=True).stdout
    clear_output(wait=True); print(time.strftime('%H:%M:%S'), '\n' + out)
    time.sleep(300)
print('finished or stopped: ', subprocess.run('bash status_linux.sh | head -3', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 6. Pack the results and download the zip (a copy also goes to Drive)
!bash pack_results_linux.sh
import glob
z = sorted(glob.glob('zkp_v4_results_*.zip'))[-1]
!cp "$z" "$BACKUP"/
files.download(z)